# Task 3: Car Price Prediction with Machine Learning

This notebook builds and evaluates regression models using the car dataset supplied for the internship task. It covers data inspection, preprocessing, feature engineering, model comparison and error analysis.

**Target:** `Selling_Price`  
**Problem type:** Supervised regression


In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore', category=FutureWarning)

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
REFERENCE_YEAR = 2026
DATA_PATH = Path('data') / 'car data.csv.gz'


## 1. Load the dataset and understand its structure
The CSV is stored as a gzip-compressed file to keep the repository lightweight. Pandas reads it directly without needing to extract it manually.


In [ ]:
df = pd.read_csv(DATA_PATH, compression='gzip')
print('Dataset shape:', df.shape)
display(df.head())
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)


## 2. Exploratory data analysis
First, inspect the selling-price distribution and its relationship with the present price. These plots help identify skew, spread and broad patterns before modelling.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Selling_Price'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of selling prices')
axes[0].set_xlabel('Selling price (source units)')
sns.scatterplot(data=df, x='Present_Price', y='Selling_Price', hue='Fuel_Type', ax=axes[1])
axes[1].set_title('Selling price vs present price')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=df, x='Fuel_Type', y='Selling_Price', ax=axes[0])
axes[0].set_title('Selling price by fuel type')
sns.boxplot(data=df, x='Transmission', y='Selling_Price', ax=axes[1])
axes[1].set_title('Selling price by transmission')
plt.tight_layout()
plt.show()


## 3. Data cleaning and feature engineering
The raw vehicle name is converted into a simple brand field using its first token. Vehicle age is calculated from the manufacturing year using 2026 as the reference year for this exercise. This is a practical baseline; a production system should use a verified manufacturer field and calculate age dynamically.


In [ ]:
data = df.copy()
text_columns = ['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission']
for col in text_columns:
    data[col] = data[col].astype('string').str.strip()

def extract_brand(name):
    cleaned = re.sub(r'[^a-zA-Z0-9 ]', ' ', str(name).lower())
    parts = cleaned.split()
    return parts[0] if parts else 'unknown'

data['Brand'] = data['Car_Name'].map(extract_brand)
data['Car_Age'] = REFERENCE_YEAR - pd.to_numeric(data['Year'], errors='coerce')
display(data[['Car_Name', 'Brand', 'Year', 'Car_Age']].head(10))
print('Brand categories:', data['Brand'].nunique())


## 4. Define features and split the data
The target column is kept separate. The original name and year are not used directly after the engineered features are created. A fixed random state makes the train/test split reproducible.


In [ ]:
numeric_features = ['Car_Age', 'Present_Price', 'Driven_kms', 'Owner']
categorical_features = ['Brand', 'Fuel_Type', 'Selling_type', 'Transmission']
feature_columns = numeric_features + categorical_features
X = data[feature_columns]
y = data['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f'Training rows: {len(X_train)}')
print(f'Test rows: {len(X_test)}')


## 5. Preprocessing pipeline
Numeric columns are median-imputed and standardised. Categorical columns are filled with the most common value and one-hot encoded. `handle_unknown='ignore'` allows the pipeline to process categories it did not see during training. The transformations are fitted only on training data.


In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])
print('Preprocessing steps are ready.')


## 6. Train and compare regression models
All three estimators use the same training and test records. The comparison uses MAE, RMSE and R². Lower MAE/RMSE and higher R² indicate better performance on this test split.


In [ ]:
models = {
    'Random Forest': RandomForestRegressor(n_estimators=400, min_samples_leaf=1, max_features=1.0, random_state=RANDOM_STATE, n_jobs=-1),
    'Extra Trees': ExtraTreesRegressor(n_estimators=400, min_samples_leaf=1, max_features=1.0, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=2, loss='huber', random_state=RANDOM_STATE)
}
trained = {}
rows = []
for model_name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', estimator)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    trained[model_name] = (pipeline, predictions)
    rows.append({
        'Model': model_name,
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': np.sqrt(mean_squared_error(y_test, predictions)),
        'R2': r2_score(y_test, predictions)
    })
results = pd.DataFrame(rows).sort_values('RMSE').reset_index(drop=True)
display(results.round(3))


## 7. Select the best model and inspect predictions
The model with the lowest test RMSE is selected for the remaining diagnostic plots. This is a simple comparison on one hold-out split, so the score should not be treated as a guarantee of future performance.


In [ ]:
best_name = results.loc[0, 'Model']
best_model, best_predictions = trained[best_name]
mae = mean_absolute_error(y_test, best_predictions)
rmse = np.sqrt(mean_squared_error(y_test, best_predictions))
r2 = r2_score(y_test, best_predictions)
print('Selected model:', best_name)
print(f'MAE:  {mae:.3f}')
print(f'RMSE: {rmse:.3f}')
print(f'R2:   {r2:.3f}')

prediction_results = pd.DataFrame({
    'Actual': y_test.to_numpy(),
    'Predicted': best_predictions,
})
prediction_results['Absolute_Error'] = (prediction_results['Actual'] - prediction_results['Predicted']).abs()
display(prediction_results.head(12).round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_predictions, ax=ax)
low = min(float(y_test.min()), float(best_predictions.min()))
high = max(float(y_test.max()), float(best_predictions.max()))
ax.plot([low, high], [low, high], '--', color='red', label='Perfect prediction')
ax.set_xlabel('Actual selling price')
ax.set_ylabel('Predicted selling price')
ax.set_title(f'Actual vs predicted – {best_name}')
ax.legend()
plt.tight_layout()
plt.show()


## 8. Residual analysis
A residual is the actual price minus the predicted price. Residuals centred around zero without a clear pattern are generally desirable. Large residuals indicate examples where the model struggled.


In [ ]:
residuals = y_test.to_numpy() - best_predictions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(residuals, kde=True, ax=axes[0])
axes[0].set_title('Residual distribution')
axes[0].set_xlabel('Actual - predicted')
sns.scatterplot(x=best_predictions, y=residuals, ax=axes[1])
axes[1].axhline(0, linestyle='--', color='red')
axes[1].set_title('Residuals vs predicted price')
axes[1].set_xlabel('Predicted selling price')
axes[1].set_ylabel('Residual')
plt.tight_layout()
plt.show()


## 9. Feature importance
Permutation importance measures the change in test-set MAE when a feature's values are shuffled. It is model-agnostic and helps show which available inputs matter most to the fitted pipeline. Importance is not proof of causation.


In [ ]:
importance = permutation_importance(
    best_model, X_test, y_test, n_repeats=20,
    random_state=RANDOM_STATE, scoring='neg_mean_absolute_error', n_jobs=-1
)
importance_df = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': importance.importances_mean,
    'Std': importance.importances_std
}).sort_values('Importance', ascending=False)
display(importance_df.round(4))
sns.barplot(data=importance_df, x='Importance', y='Feature')
plt.title('Permutation feature importance')
plt.tight_layout()
plt.show()


## 10. Example prediction
The example below shows how a new record can be passed through the selected pipeline. Values must use the same source units and categories as the training data. Brand extraction is performed using the same helper function as above.


In [ ]:
example_car = pd.DataFrame([{
    'Car_Age': REFERENCE_YEAR - 2017,
    'Present_Price': 8.50,
    'Driven_kms': 25000,
    'Owner': 0,
    'Brand': 'maruti',
    'Fuel_Type': 'Petrol',
    'Selling_type': 'Dealer',
    'Transmission': 'Manual'
}], columns=feature_columns)
estimated_price = best_model.predict(example_car)[0]
print(f'Illustrative estimated selling price: {estimated_price:.2f} (source price units)')


## 11. Conclusion and limitations
The workflow demonstrates how a regression model can estimate used-car selling prices from vehicle details. Random Forest is the strongest model on this particular held-out split. The data is small and historical, and important details such as vehicle condition, location, accident history and service records are absent. `Present_Price` is also a highly informative input and must be available with a consistent meaning when the model is used. A real valuation system should be validated on recent, independent listings and should communicate its estimate as guidance rather than a guaranteed market price.
